# AFL Win Probability — ML Workshop

A step-by-step walkthrough of building an in-game win probability model for AFL, from a simple lookup table to gradient boosting.

**Data:** AFL 2012–2024 (train: 2012–2021 · val: 2022–2023).
*Final evaluation on 2024–2025 is held back and will be run at the end of the workshop.*

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    log_loss, brier_score_loss, roc_auc_score,
)
from pathlib import Path

gold = pd.read_csv(Path('..') / 'data' / 'gold' /
                   '2012-2025' / 'model_dataset.csv')
gold = gold[gold['quarter'] <= 4].reset_index(drop=True)   # drop extra time
print(f'Rows: {len(gold):,}   |   Games: {gold["match_id"].nunique():,}')
gold.head(8)

## 1. The Dataset

**How it was built:**

1. **Scrape** — Every score event (goal or point) is timestamped for every AFL game since 2012. Each row is one team's state at one moment.
2. **Symmetry** — Each game is duplicated: once from the home team's perspective, once from the away team's. This keeps the dataset balanced at ~50% win rate — meaning a trivial model that always predicts "win" scores ~50% accuracy, so any accuracy above 50% represents genuine predictive signal.
3. **Features & target** — `score_diff` = this team's score minus opponent's (positive = leading); `won` = 1 if this team won, 0 otherwise.

In [ ]:
cols = ['match_id', 'quarter', 'minute_in_qtr', 'elapsed_secs_in_qtr',
        'score_diff', 'relative_margin', 'won', 'is_train', 'is_val']
gold[cols].sample(12, random_state=1)

In [ ]:
# Pre-compute derived columns used in later models
gold['score_diff_x_elapsed'] = gold['score_diff'] * gold['elapsed_secs_in_qtr']
gold['abs_relative_margin'] = gold['relative_margin'].abs()
gold['total_score'] = gold['home_score'] + gold['away_score']

train_mask = gold['is_train']
val_mask   = gold['is_val']
print('Done.')

In [ ]:
def metrics(name, y, probs):
    """6 evaluation metrics in a single-row DataFrame."""
    p = np.clip(probs, 1e-7, 1 - 1e-7)
    preds = (p >= 0.5).astype(int)
    return pd.DataFrame([{
        'model':     name,
        'accuracy':  round(accuracy_score(y, preds), 4),
        'precision': round(precision_score(y, preds, zero_division=0), 4),
        'recall':    round(recall_score(y, preds, zero_division=0), 4),
        'log_loss':  round(log_loss(y, p), 4),
        'brier':     round(brier_score_loss(y, p), 4),
        'auc_roc':   round(roc_auc_score(y, p), 4),
    }])

---
## 2. Model A — Conditional Means (Non-parametric Baseline)

The simplest approach: **look up the historical win rate** for teams in a given situation.

For each **(quarter, score bracket, minute)** in the training data, compute the fraction of times teams with that profile won. No fitting, no assumptions — just counting.

- **Score brackets:** 1-pt bins for the close-game range (−11 to +12 pts, i.e. within ~2 goals); 6-pt bins for blowout territory on either side — where data is thin and 1-pt precision would create noise
- 4 quarters × 42 brackets × 41 minutes = ~6,888 possible cells

In [ ]:
# ── Model A — your implementation ───────────────────────────────────────────
# Step 1: define score bins
# Hint: use the edges below (already defined for you) with pd.cut()
_EDGES = (
    [-np.inf, -59, -53, -47, -41, -35, -29, -23, -17]  # outer negative (6-pt steps)
    + list(range(-11, 14))                               # 1-pt core: -11 … +13
    + [19, 25, 31, 37, 43, 49, 55, 61, np.inf]          # outer positive (6-pt steps)
)
_LABELS = (
    ['<-59', '-59 to -54', '-53 to -48', '-47 to -42', '-41 to -36',
     '-35 to -30', '-29 to -24', '-23 to -18', '-17 to -12']
    + [str(x) for x in range(-11, 13)]
    + ['13 to 18', '19 to 24', '25 to 30', '31 to 36', '37 to 42',
       '43 to 48', '49 to 54', '55 to 60', '61+']
)

def add_bracket(df):
    # TODO: add a 'score_bracket' column to df using pd.cut with _EDGES and _LABELS
    raise NotImplementedError("implement add_bracket")

# Step 2: build lookup table from training data
# _train_a = ...
# lookup = ...  groupby (quarter, score_bracket, minute_in_qtr), take mean of 'won'
# lookup_df = lookup.reset_index().rename(columns={'won': 'prob'})

# Step 3: implement the prediction function
def model_a_probs(mask):
    # Merge gold[mask] with lookup_df on (quarter, score_bracket, minute_in_qtr)
    # Fill missing cells with 0.5
    # Return a numpy array of probabilities
    raise NotImplementedError("implement model_a_probs")

# print(f'Cells with data: ...')

In [ ]:
# Optional: display a Q4 win-probability lookup table once your lookup is built
# _DISP_MINS = list(range(0, 25, 5)) + list(range(25, 36)) + [40]
# q4 = (
#     lookup.xs(4, level='quarter')
#           .unstack('minute_in_qtr')
#           .reindex(index=_LABELS, columns=range(0, 41))
#           .round(3)
# )
# show_rows = (['−35 to −30', '−29 to −24', '−23 to −18', '−17 to −12']
#              + [str(x) for x in range(-6, 7)]
#              + ['13 to 18', '19 to 24', '25 to 30', '31 to 36'])
# show_rows = [r for r in show_rows if r in q4.index]
# q4.loc[show_rows, _DISP_MINS]

### Flaws of Model A

1. **Sparse cells default to 0.5.** If a (bracket, minute) combo never appeared in training, the model outputs 50% — badly wrong for extreme scores (e.g. up 60 pts in Q4 minute 35).
2. **No information sharing.** Each cell is estimated in isolation. Being up 11 pts at minute 5 tells the model nothing about being up 12 pts at minute 5 — even though those situations are almost identical.
3. **Non-monotonic predictions.** The lookup table can produce win probabilities that go *down* as the score margin increases, purely due to noise in thin cells.

Let's check how many validation rows get the 0.5 fallback:

In [ ]:
# TODO: check how many val rows default to 0.5 (i.e. had no training data for their cell)
# val_a = add_bracket(gold[gold['is_val']]).merge(
#     lookup_df, on=['quarter', 'score_bracket', 'minute_in_qtr'], how='left'
# )
# missing = val_a['prob'].isna().sum()
# print(f'Val rows defaulting to 0.5: {missing:,}  ({100*missing/len(val_a):.1f}%)')

In [ ]:
# TODO: evaluate Model A on train and val
# pd.concat([
#     metrics('A (train)', gold.loc[train_mask, 'won'].values, model_a_probs(train_mask)),
#     metrics('A (val)',   gold.loc[val_mask,   'won'].values, model_a_probs(val_mask)),
# ]).reset_index(drop=True)

<details>
<summary><strong>Reference — Model A expected metrics</strong></summary>

| split | accuracy | precision | recall | log_loss | brier | auc_roc |
|-------|----------|-----------|--------|----------|-------|---------|
| train | 0.7861   | 0.7921    | 0.7718 | 0.4138   | 0.1385 | 0.8846 |
| val   | 0.7504   | 0.7537    | 0.7389 | 0.4744   | 0.1576 | 0.8511 |

Val rows defaulting to 0.5: **222 (0.2%)**
</details>

### Reading the metrics

| Metric | What it measures | Better when |
|--------|-----------------|-------------|
| **Accuracy** | Fraction of correct win/loss predictions (threshold 0.5) | ↑ higher |
| **Precision** | Of predicted wins, how many were actual wins | ↑ higher |
| **Recall** | Of actual wins, how many did we predict correctly | ↑ higher |
| **Log-loss** | Average log-penalty for wrong confidence (heavily penalises confident mistakes) | ↓ lower |
| **Brier score** | Mean squared error of probabilities against 0/1 outcomes | ↓ lower |
| **AUC-ROC** | Probability that the model ranks a random win above a random loss | ↑ higher |

Log-loss and Brier score are the most informative here — they evaluate probability *quality*, not just which side of 50% the model lands on.

---
## 3. Model B — Logistic Regression on `score_diff`

Instead of a lookup table, fit a **smooth parametric model**. One logistic regression per quarter — two steps:

```
log-odds  =  a  +  b × score_diff
p(win)    =  1 / (1 + exp(−log-odds))
```

**What are odds and log-odds?**
- **Odds** = p / (1 − p). If you win 2 in every 3 games, odds = 2:1 = 2.0. Odds > 1 means favoured; odds < 1 means underdog.
- **Log-odds** (logit) = log(odds). It maps (0, 1) probability to (−∞, +∞), letting us model it with a straight line in score_diff.

Putting it together:  `p(win) = 1 / (1 + exp(−(a + b × score_diff)))`

The coefficient `b` is estimated from **all rows in that quarter**.

In [ ]:
# Helper functions — these will work once you implement the fitting below

logit_models   = {}   # (name, quarter) -> fitted statsmodels Logit result
logit_features = {}   # (name, quarter) -> feature list


def fit_logit(name, features):
    """Fit one logistic regression per quarter using training data."""
    # TODO: for each quarter q in [1, 2, 3, 4]:
    #   1. filter gold to is_train AND quarter == q
    #   2. build X with sm.add_constant(train[features], has_constant='add')
    #   3. fit sm.Logit(train['won'], X).fit(disp=False)
    #   4. store in logit_models[(name, q)] and logit_features[(name, q)]
    raise NotImplementedError("implement fit_logit")


def logit_probs(name, mask=None):
    """Return predicted probabilities for all rows (optionally filtered by mask)."""
    p = np.full(len(gold), np.nan)
    for q in [1, 2, 3, 4]:
        qmask = gold['quarter'] == q
        X = sm.add_constant(
            gold.loc[qmask, logit_features[(name, q)]], has_constant='add')
        p[qmask.values] = logit_models[(name, q)].predict(X)
    return p if mask is None else p[mask.values]


def show_coefs(name):
    """Return a DataFrame of coefficients and t-stats for each quarter."""
    rows = []
    for q in [1, 2, 3, 4]:
        m = logit_models[(name, q)]
        for var in m.params.index:
            rows.append({'Q': q, 'variable': var,
                         'coef':  round(m.params[var],  5),
                         'tstat': round(m.tvalues[var], 2)})
    return pd.DataFrame(rows)


print('Helpers ready.')

In [ ]:
# TODO: fit Model B (single feature: score_diff) and display coefficients
# fit_logit('B', ['score_diff'])
# show_coefs('B')

<details>
<summary><strong>Reference — Model B expected coefficients</strong></summary>

| Q | variable | coef | t-stat |
|---|----------|------|--------|
| 1 | const      | −0.01539 | −2.57  |
| 1 | score_diff |  0.06960 | 112.72 |
| 2 | const      | −0.02041 | −2.97  |
| 2 | score_diff |  0.08039 | 168.72 |
| 3 | const      | −0.02924 | −3.55  |
| 3 | score_diff |  0.10168 | 177.24 |
| 4 | const      | −0.05615 | −4.98  |
| 4 | score_diff |  0.16900 | 149.19 |

Notice `b` (score_diff coefficient) increases each quarter — a 1-goal lead means more as the game progresses.
</details>

### Interpreting the coefficients

| Term | Meaning |
|------|---------|
| **const (a)** | Log-odds of winning when score_diff = 0 (tied). |
| **b (score_diff)** | Each additional point increases log-odds by b. Odds multiply by exp(b) per point. |
| **t-stat** | b divided by its standard error. |

**Statistical significance:** |t| > 2 means the coefficient is significantly non-zero. Ignore coefficients with |t| < 2 when interpreting results.

**Why is `a` (const) negative?** At score_diff = 0 the game is tied — a draw counts as 0, so the probability of *winning* (not drawing) when tied is slightly < 50%, hence log-odds < 0.

**Converting to probability:** given score_diff = d,  p(win) = 1 / (1 + exp(-(a + b * d)))

In [ ]:
# TODO: show win probability at the START of each quarter for a range of score differences
# score_diffs = [-18, -12, -6, 0, 6, 12, 18, 24]
# prob_rows = []
# for d in score_diffs:
#     row = {'score_diff': d, 'goals_ahead': d // 6}
#     for q in [1, 2, 3, 4]:
#         m = logit_models[('B', q)]
#         a, b = m.params['const'], m.params['score_diff']
#         row[f'Q{q}'] = round(1 / (1 + np.exp(-(a + b * d))), 3)
#     prob_rows.append(row)
# pd.DataFrame(prob_rows).set_index(['score_diff', 'goals_ahead'])

In [ ]:
# TODO: evaluate Model B on train and val
# pd.concat([
#     metrics('B (train)', gold.loc[train_mask, 'won'].values, logit_probs('B', train_mask)),
#     metrics('B (val)',   gold.loc[val_mask,   'won'].values, logit_probs('B', val_mask)),
# ]).reset_index(drop=True)

<details>
<summary><strong>Reference — Model B expected metrics</strong></summary>

| split | accuracy | precision | recall | log_loss | brier | auc_roc |
|-------|----------|-----------|--------|----------|-------|---------|
| train | 0.7836   | 0.7933    | 0.7629 | 0.4209   | 0.1405 | 0.8811 |
| val   | 0.7533   | 0.7617    | 0.7326 | 0.4620   | 0.1564 | 0.8530 |
</details>

### Val vs Test — Read This Before Proceeding

Throughout this workshop you will see **val metrics** (2022–2023). You are allowed to look at these while building and comparing models.

**However:** every time you look at val performance and use it to make a decision you are implicitly fitting to val. This is called **p-hacking**, and it erodes your budget.

The rules:
- **Val (2022–2023):** use freely to guide development and compare approaches.
- **Test (held-out):** evaluated **once**, at the very end. Never tune to it. Never peek.

Treat val as "informative but cheap". Treat test as "one shot".

### Model A vs Model B — What Changes?

| | Model A | Model B |
|--|---------|----------|
| **Type** | Lookup table | Logistic regression |
| **Information sharing** | Each (bracket, minute) cell estimated independently | One b estimated from ALL rows |
| **Sparse situations** | Defaults to 0.5 | Extrapolates smoothly from the fitted curve |
| **Domain knowledge** | Manual 6-pt bins | None — takes raw score_diff directly |

**Is B 'smoothed'?** Yes. Because b is estimated from every row in the quarter, a team up 12 pts at minute 2 and a team up 12 pts at minute 38 both inform the same coefficient. This is more data-efficient than isolating each (bracket, minute) cell.

---
## 4. Model C — Adding Elapsed Time

**Intuition:** being up 10 pts with 1 minute left is very different from being up 10 pts with 15 minutes left. We add `elapsed_secs_in_qtr` and an interaction `score_diff × elapsed`.

The interaction is key: it lets the *slope* of score_diff change as the quarter progresses. As time runs out, each point becomes more decisive.

In [ ]:
# TODO: fit Model C with features: score_diff, elapsed_secs_in_qtr, score_diff_x_elapsed
# fit_logit('C', ['score_diff', 'elapsed_secs_in_qtr', 'score_diff_x_elapsed'])
# show_coefs('C')

<details>
<summary><strong>Reference — Model C expected coefficients</strong></summary>

| Q | variable | coef | t-stat |
|---|----------|------|--------|
| 1 | const                  | −0.01421 | −1.26  |
| 1 | score_diff             |  0.06876 |  39.35 |
| 1 | elapsed_secs_in_qtr   | ~0.00000 |  −0.12 |
| 1 | score_diff_x_elapsed  | ~0.00000 |   0.52 |
| 2 | const                  | −0.01843 | −1.44  |
| 2 | score_diff             |  0.07230 |  77.97 |
| 2 | elapsed_secs_in_qtr   | ~0.00000 |  −0.18 |
| 2 | score_diff_x_elapsed  |  0.00001 |   9.92 |
| 3 | const                  | −0.02577 | −1.70  |
| 3 | score_diff             |  0.08749 |  85.10 |
| 3 | elapsed_secs_in_qtr   | ~0.00000 |  −0.28 |
| 3 | score_diff_x_elapsed  |  0.00002 |  15.64 |
| 4 | const                  | −0.03217 | −1.66  |
| 4 | score_diff             |  0.10332 |  64.88 |
| 4 | elapsed_secs_in_qtr   | −0.00003 |  −1.66 |
| 4 | score_diff_x_elapsed  |  0.00010 |  41.04 |

The `score_diff_x_elapsed` interaction becomes significant (|t| > 2) in Q2, Q3, Q4 — meaning elapsed time only meaningfully amplifies the score margin from Q2 onwards.
</details>

In [ ]:
# TODO: build a Q4 win-probability table (score_diff × minute) for Model C
# _DISP_SCORES = [-30, -24, -18, -12] + list(range(-6, 7)) + [13, 19, 25, 31]
# _DISP_MINS   = list(range(0, 25, 5)) + list(range(25, 36)) + [40]
# rows = []
# for sd in _DISP_SCORES:
#     row = {'score_diff': sd}
#     for mn in _DISP_MINS:
#         elapsed = mn * 60
#         m  = logit_models[('C', 4)]
#         pr = m.params
#         lp = (pr['const']
#               + pr['score_diff']            * sd
#               + pr['elapsed_secs_in_qtr']   * elapsed
#               + pr['score_diff_x_elapsed']  * sd * elapsed)
#         row[f'{mn}m'] = round(1 / (1 + np.exp(-lp)), 3)
#     rows.append(row)
# pd.DataFrame(rows).set_index('score_diff')

### Interpreting the interaction term (Q4)

The key new coefficient is `score_diff_x_elapsed`. Its positive sign means: **the effect of a point lead grows as the quarter progresses**.

The marginal contribution of one extra point at time *t* seconds is:

    b_score_diff + b_interaction × t

In Q4 at minute 0 (t = 0) this is just `b_score_diff ≈ 0.10`. By minute 30 (t = 1800 s) it has grown by `b_interaction × 1800`. This is exactly what you see in the probability table: a 3-pt lead flips from ~57% at the start of the quarter to above 90% with 5 minutes remaining.

> **If you want to understand interaction terms formally**, search "interaction effects in logistic regression." The core idea: `score_diff × elapsed` is the product of the two variables, which allows the *slope* of one to be a function of the other.

---

In [ ]:
# TODO: evaluate Model C on train and val
# pd.concat([
#     metrics('C (train)', gold.loc[train_mask, 'won'].values, logit_probs('C', train_mask)),
#     metrics('C (val)',   gold.loc[val_mask,   'won'].values, logit_probs('C', val_mask)),
# ]).reset_index(drop=True)

<details>
<summary><strong>Reference — Model C expected metrics</strong></summary>

| split | accuracy | precision | recall | log_loss | brier | auc_roc |
|-------|----------|-----------|--------|----------|-------|---------|
| train | 0.7836   | 0.7933    | 0.7629 | 0.4187   | 0.1401 | 0.8818 |
| val   | 0.7533   | 0.7617    | 0.7326 | 0.4586   | 0.1557 | 0.8542 |

Accuracy/precision/recall barely change from Model B — the main gain is in log-loss and Brier (probability quality), not in which side of 50% each row lands.
</details>

### Discussion question

**Q: Accuracy, precision, and recall barely change from Model B to Model C, even though log-loss and Brier improve. Why?**

Accuracy, precision, and recall all depend on converting a probability to a hard prediction via a 0.5 threshold. If a team was 62% likely to win under Model B and 68% likely under Model C, both models predict "win" — no reclassification happens. The only rows that change their prediction label are those where the probability crosses 50%.

Log-loss and Brier score see the improvement because they penalise *how far* a probability is from the true outcome. A probability moving from 62% to 68% when the team won is a real improvement in those metrics, even though it doesn't change the accuracy count.

---
## 5. LightGBM — Beyond Linear Models

All models so far are linear in log-odds space. **LightGBM** is a gradient-boosted decision tree: it builds many shallow trees sequentially, each correcting the residuals of the previous.

Key properties:
- Captures **non-linear relationships** and **interactions** automatically — no manual feature engineering
- We give it `score_diff` and `elapsed_secs_in_qtr` — the same two features as Model C — so we can compare on equal footing
- Hyperparameters control complexity: `max_depth`, `num_leaves`, regularisation, subsampling

> **There are many other models worth trying:** Random Forests, XGBoost, CatBoost, model blending. The takeaway from this workshop: `score_diff` is so dominant that even simple logistic regression is hard to beat substantially.

In [ ]:
LGBM_FEATURES = ['score_diff', 'elapsed_secs_in_qtr']
lgbm_models   = {}

PARAMS = dict(
    objective='binary', n_estimators=1000, learning_rate=1e-2,
    reg_lambda=1e-3, max_depth=12, colsample_bytree=0.75,
    subsample=0.75, num_leaves=31, min_child_samples=20,
    random_state=42, verbose=-1,
)

# TODO: for each quarter q in [1, 2, 3, 4]:
#   filter gold to is_train AND quarter == q
#   fit LGBMClassifier(**PARAMS) on LGBM_FEATURES, target 'won'
#   store in lgbm_models[q]
# print('All LGBM models fitted.')

In [ ]:
# TODO: show feature importance by gain for each quarter
# imp_rows = []
# for q in [1, 2, 3, 4]:
#     gains = lgbm_models[q].booster_.feature_importance(importance_type='gain')
#     for feat, g in zip(LGBM_FEATURES, gains):
#         imp_rows.append({'quarter': q, 'feature': feat, 'gain': round(float(g), 0)})
# pd.DataFrame(imp_rows).pivot(index='feature', columns='quarter', values='gain')

In [ ]:
def lgbm_probs(mask):
    p = np.full(len(gold), np.nan)
    for q in [1, 2, 3, 4]:
        qmask = gold['quarter'] == q
        p[qmask.values] = lgbm_models[q].predict_proba(
            gold.loc[qmask, LGBM_FEATURES])[:, 1]
    return p[mask.values]


# TODO: evaluate LGBM on train and val
# pd.concat([
#     metrics('LGBM (train)', gold.loc[train_mask, 'won'].values, lgbm_probs(train_mask)),
#     metrics('LGBM (val)',   gold.loc[val_mask,   'won'].values, lgbm_probs(val_mask)),
# ]).reset_index(drop=True)

<details>
<summary><strong>Reference — LGBM expected metrics</strong></summary>

| split | accuracy | precision | recall | log_loss | brier | auc_roc |
|-------|----------|-----------|--------|----------|-------|---------|
| train | 0.7844   | 0.7955    | 0.7617 | 0.4125   | 0.1386 | 0.8841 |
| val   | 0.7528   | 0.7624    | 0.7297 | 0.4598   | 0.1557 | 0.8541 |

Feature importance (gain) confirms `score_diff` dominates by ~100× over `elapsed_secs_in_qtr` in every quarter. LGBM's flexibility doesn't dramatically outperform the logistic models here precisely because the signal is so concentrated in one linear feature.
</details>

---
## 6. Final Comparison — All Models (Val set)

In [ ]:
# TODO: collect val metrics from all four models into a single comparison DataFrame
# all_rows = []
# all_rows.append(metrics('A: conditional means', ...).iloc[0])
# all_rows.append(metrics('Logistic B',           ...).iloc[0])
# all_rows.append(metrics('Logistic C',           ...).iloc[0])
# all_rows.append(metrics('LGBM',                 ...).iloc[0])
# pd.DataFrame(all_rows).reset_index(drop=True)

<details>
<summary><strong>Reference — full val comparison (all models)</strong></summary>

| model | accuracy | precision | recall | log_loss | brier | auc_roc |
|-------|----------|-----------|--------|----------|-------|---------|
| A: conditional means | 0.7504 | 0.7537 | 0.7389 | 0.4744 | 0.1576 | 0.8511 |
| Logistic B           | 0.7533 | 0.7617 | 0.7326 | 0.4620 | 0.1564 | 0.8530 |
| Logistic C           | 0.7533 | 0.7617 | 0.7326 | 0.4586 | 0.1557 | 0.8542 |
| LGBM                 | 0.7528 | 0.7624 | 0.7297 | 0.4598 | 0.1557 | 0.8541 |

Key takeaway: all models are within ~0.004 AUC of each other. The signal is concentrated in `score_diff`; the main gains from more complex models are in probability calibration (log-loss / Brier), not in discrimination (AUC-ROC).
</details>

---
## 7. Where to Go From Here

**Features to try:**

- **`total_score`** — captures game pace (how many points are on the board). In a low-scoring game each point is worth more, which `relative_margin` already encodes indirectly. Adding `total_score` directly may improve calibration in extreme cases, but it is highly correlated with `relative_margin` — expect modest gains and watch for inflated coefficients.
- **One-hot encoding `home_team` / `away_team`** — each team gets its own indicator, capturing systematic home-ground advantages. With 18 teams you add ~36 binary features; each will be estimated from limited data (~150 games per team), so regularise heavily or you will overfit to training-era team quality that may not persist to val/test.

**Models to try:**

- **XGBoost / CatBoost** — alternatives to LightGBM with different regularisation
- **Random Forest** — lower variance, good calibration out of the box